In [ ]:
!pip install -q anthropic

from google.colab import drive
drive.mount("/content/drive")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.6 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import anthropic

ANTHROPIC_API_KEY = ""

assert ANTHROPIC_API_KEY.startswith("sk-ant-"), (
    "The Anthropic API key appears to be invalid."
)

claude_client = anthropic.Anthropic(
    api_key=ANTHROPIC_API_KEY
)

print("Anthropic client initialized.")

Anthropic client initialized.


In [ ]:
import json
import re
import time
from pathlib import Path

RESULT_DIR = Path(
    "/content/drive/MyDrive/cbt_results"
)

INPUT_PATH = (
    RESULT_DIR / "responses_for_judge.json"
)

OUTPUT_PATH = (
    RESULT_DIR / "rag_judge_temperature0.json"
)

assert INPUT_PATH.exists(), (
    f"Input file not found: {INPUT_PATH}"
)

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    single_turn_cases = json.load(f)

assert len(single_turn_cases) == 100, (
    f"Expected 100 cases, found {len(single_turn_cases)}."
)

for case in single_turn_cases:
    assert isinstance(
        case.get("rag_response"), str
    ) and case["rag_response"].strip(), (
        f"Missing rag_response for case {case.get('id')}."
    )

print(f"Loaded {len(single_turn_cases)} RAG responses.")
print("The original response file will be read only.")

Loaded 100 RAG responses.
The original response file will be read only.


In [ ]:
JUDGE_MODEL = "claude-sonnet-4-5"

JUDGE_DIMS = [
    "CBT_Accuracy",
    "Empathy",
    "Relevance",
    "Clinical_Appropriateness",
    "Overall",
]

KEY_ALIASES = {
    "CBT_Accuracy": [
        "CBT_Accuracy",
        "CBT Accuracy",
        "CBTAccuracy",
        "cbt_accuracy",
    ],
    "Empathy": [
        "Empathy",
        "empathy",
    ],
    "Relevance": [
        "Relevance",
        "relevance",
    ],
    "Clinical_Appropriateness": [
        "Clinical_Appropriateness",
        "Clinical Appropriateness",
        "ClinicalAppropriateness",
        "clinical_appropriateness",
    ],
    "Overall": [
        "Overall",
        "overall",
        "Average",
        "average",
    ],
}

JUDGE_PROMPT_TEMPLATE = """
You are an expert evaluator of CBT
(Cognitive Behavioral Therapy) chatbot responses.

Evaluate the following response to a client's message
on these 4 dimensions, scoring each from 1-10:

1. CBT_Accuracy: Does the response correctly apply CBT
techniques such as Socratic questioning, cognitive
restructuring, thought challenging, identifying automatic
thoughts, or behavioral activation?

2. Empathy: Is the response warm, non-judgmental,
validating, and emotionally attuned?

3. Relevance: Does the response directly address the
client's message without going off-topic?

4. Clinical_Appropriateness: Is the response professionally
appropriate for a CBT support context, avoiding diagnosis,
overclaiming, unsafe advice, or replacing professional care?

Client message:
{user_input}

Response to evaluate:
{response}

Return ONLY valid JSON.
Use EXACTLY these keys:

{{
  "CBT_Accuracy": <number from 1 to 10>,
  "Empathy": <number from 1 to 10>,
  "Relevance": <number from 1 to 10>,
  "Clinical_Appropriateness": <number from 1 to 10>,
  "Overall": <average of the four numeric scores>,
  "Reasoning": "<one concise sentence>"
}}
""".strip()

In [ ]:
def extract_json(raw_text):
    raw_text = (
        raw_text
        .strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    match = re.search(
        r"\{.*\}",
        raw_text,
        re.DOTALL,
    )

    if not match:
        raise ValueError(
            f"No JSON object found: {raw_text[:300]}"
        )

    return json.loads(match.group())


def normalize_judge_scores(parsed):
    normalized = {}

    for target_key, aliases in KEY_ALIASES.items():
        value = None

        for alias in aliases:
            if alias in parsed:
                value = parsed[alias]
                break

        if value is None:
            raise ValueError(
                f"Missing judge field: {target_key}"
            )

        normalized[target_key] = float(value)

    reasoning = (
        parsed.get("Reasoning")
        or parsed.get("reasoning")
        or ""
    )

    normalized["Reasoning"] = reasoning

    for dimension in JUDGE_DIMS:
        score = normalized[dimension]

        if not 1.0 <= score <= 10.0:
            raise ValueError(
                f"Invalid {dimension} score: {score}"
            )

    return normalized


def judge_rag_response(
    user_input,
    response,
    max_retries=3,
):
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        user_input=user_input,
        response=response,
    )

    last_error = None

    for attempt in range(max_retries):
        try:
            message = claude_client.messages.create(
                model=JUDGE_MODEL,
                max_tokens=400,
                temperature=0,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
            )

            raw_text = message.content[0].text

            parsed = extract_json(raw_text)

            return normalize_judge_scores(parsed)

        except Exception as error:
            last_error = error

            print(
                f"Attempt {attempt + 1} failed: {error}"
            )

            time.sleep(2 + attempt * 2)

    raise RuntimeError(
        f"Judge failed after {max_retries} attempts: "
        f"{last_error}"
    )

In [ ]:
if OUTPUT_PATH.exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        temperature0_results = json.load(f)

    print(
        f"Resuming from {len(temperature0_results)} "
        "completed cases."
    )
else:
    temperature0_results = []

completed_ids = {
    record["id"]
    for record in temperature0_results
}

for index, case in enumerate(
    single_turn_cases,
    start=1,
):
    case_id = case["id"]

    if case_id in completed_ids:
        print(
            f"[{index:03d}/100] Case {case_id} "
            "already completed."
        )
        continue

    scores = judge_rag_response(
        user_input=case["user_input"],
        response=case["rag_response"],
    )

    result_record = {
        "id": case_id,
        "category": case["category"],
        "judge_model": JUDGE_MODEL,
        "temperature": 0,
        "rag_judge_scores": scores,
    }

    temperature0_results.append(result_record)
    completed_ids.add(case_id)

    print(
        f"[{index:03d}/100] "
        f"{case['category']:24s} "
        f"Overall={scores['Overall']:.2f}"
    )

    if (
        len(temperature0_results) % 5 == 0
        or len(temperature0_results) == 100
    ):
        temperature0_results.sort(
            key=lambda record: record["id"]
        )

        with open(
            OUTPUT_PATH,
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(
                temperature0_results,
                f,
                ensure_ascii=False,
                indent=2,
            )

        print(
            f"Checkpoint saved: "
            f"{len(temperature0_results)}/100"
        )

    time.sleep(0.3)

assert len(temperature0_results) == 100

print("\nTemperature-zero RAG evaluation completed.")
print("New result file:", OUTPUT_PATH)
print("Original result files were not modified.")

[001/100] anxiety                  Overall=8.75
[002/100] anxiety                  Overall=8.00
[003/100] anxiety                  Overall=8.25
[004/100] anxiety                  Overall=9.00
[005/100] anxiety                  Overall=8.50
Checkpoint saved: 5/100
[006/100] negative_self_talk       Overall=9.00
[007/100] negative_self_talk       Overall=9.00
[008/100] negative_self_talk       Overall=8.50
[009/100] negative_self_talk       Overall=8.00
[010/100] negative_self_talk       Overall=8.75
Checkpoint saved: 10/100
[011/100] perfectionism            Overall=8.25
[012/100] perfectionism            Overall=8.00
[013/100] perfectionism            Overall=8.25
[014/100] perfectionism            Overall=9.00
[015/100] perfectionism            Overall=9.00
Checkpoint saved: 15/100
[016/100] catastrophizing          Overall=8.00
[017/100] catastrophizing          Overall=9.25
[018/100] catastrophizing          Overall=9.00
[019/100] catastrophizing          Overall=9.25
[020/100] cata

In [ ]:
import numpy as np
import pandas as pd

summary = {}

for dimension in JUDGE_DIMS:
    values = [
        record["rag_judge_scores"][dimension]
        for record in temperature0_results
    ]

    summary[dimension] = {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
    }

summary_df = pd.DataFrame(summary).T
summary_df.index.name = "Dimension"

display(summary_df.round(3))

assert all(
    record["temperature"] == 0
    for record in temperature0_results
)

assert len({
    record["id"]
    for record in temperature0_results
}) == 100

print("PASS: 100 unique RAG responses were evaluated at temperature=0.")

,mean,std,minimum,maximum
Dimension,,,,
CBT_Accuracy,7.800,0.812,4.0,9.0
Empathy,7.930,0.852,5.0,9.0
Relevance,9.450,0.536,8.0,10.0
Clinical_Appropriateness,8.870,0.541,7.0,10.0
Overall,8.512,0.527,6.5,9.5


PASS: 100 unique RAG responses were evaluated at temperature=0.
